
# StudyChat — Dialogue Act labeling from **prompt** using natural-language label descriptions (train split)

This notebook:
- loads the gated HF dataset `wmcnicho/StudyChat` (using `HF_TOKEN`),
- assumes **all data are in `train`**,
- classifies **only** the `prompt` field,
- uses **natural-language descriptions** for the 41 SwDA labels (better zero-shot semantics),
- maps predictions back to the short **SwDA codes**,
- saves:
  - **JSON**: original objects + `da_label` (code) + `da_name` (human label) + `da_score`
  - **CSV**: selected flat columns + the same prediction fields.

> Tip: If you're just testing, set `MAX_SAMPLES` to a small number and reduce `BATCH_SIZE` if you run out of memory.


In [1]:

# Optional installs if running locally
%pip install -q datasets transformers torch tqdm python-dotenv pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:

import os, json
from typing import List, Dict, Any, Tuple
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import pipeline
import torch
import pandas as pd
from tqdm import tqdm

# Load env for gated dataset
load_dotenv('.env')
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    print('⚠️  HF_TOKEN not set. Add it to your .env to access gated datasets.')

DATASET_NAME = "wmcnicho/StudyChat"
SPLIT = "train"  # per your spec

# Inference config
BATCH_SIZE = 16
DEVICE = 0 if torch.cuda.is_available() else -1
TOP_K = 1
MAX_SAMPLES = None  # set to an int for quick tests

# Outputs
OUT_JSON = "studychat_train_with_da_desc.json"
OUT_CSV = "studychat_train_with_da_desc.csv"


/opt/homebrew/Caskroom/miniconda/base/envs/studychat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# 41 SwDA labels: short code -> (human name, description phrase)
# Descriptions are phrased to make sense in: "This is a {label_description}."
DA_DEF: Dict[str, Tuple[str, str]] = {
    "sd": ("Statement-non-opinion", "factual statement providing information"),
    "b": ("Acknowledge (Backchannel)", "short acknowledgement such as okay, uh-huh, or I see"),
    "sv": ("Statement-opinion", "opinion or subjective statement"),
    "%": ("Uninterpretable", "uninterpretable or garbled utterance"),
    "aa": ("Agree/Accept", "explicit agreement or acceptance of a prior statement"),
    "ba": ("Appreciation", "expression of gratitude or appreciation"),
    "qy": ("Yes-No-Question", "yes-no question expecting a yes or no answer"),
    "ny": ("Yes Answers", "yes answer"),
    "fc": ("Conventional-closing", "conversational closing such as goodbye or see you"),
    "qw": ("Wh-Question", "wh-question beginning with what, why, how, where, when, or who"),
    "nn": ("No Answers", "no answer"),
    "bk": ("Response Acknowledgement", "acknowledgement of receiving or understanding a response"),
    "h": ("Hedge", "hedge or uncertainty marker such as maybe, I think, or probably"),
    "qy^d": ("Declarative Yes-No-Question", "declarative form that functions as a yes-no question"),
    "bh": ("Backchannel in Question Form", "backchannel posed in question form"),
    "^q": ("Quotation", "quotation or reported speech"),
    "bf": ("Summarize/Reformulate", "summary or reformulation of prior content"),
    "fo_o_fw_by_bc": ("Other", "other or miscellaneous dialogue act"),
    "na": ("Affirmative Non-yes Answers", "affirmative answer that is not an explicit yes"),
    "ad": ("Action-directive", "instruction or directive to perform an action"),
    "^2": ("Collaborative Completion", "collaborative completion of another speaker's utterance"),
    "b^m": ("Repeat-phrase", "repetition of a phrase"),
    "qo": ("Open-Question", "open question not strictly yes-no or wh-form"),
    "qh": ("Rhetorical-Question", "rhetorical question"),
    "^h": ("Hold Before Answer/Agreement", "hold or pause before answering or agreeing"),
    "ar": ("Reject", "explicit rejection or disagreement"),
    "ng": ("Negative Non-no Answers", "negative answer that is not an explicit no"),
    "br": ("Signal-non-understanding", "signal of non-understanding such as pardon or I didn't get that"),
    "no": ("Other Answers", "other type of answer"),
    "fp": ("Conventional-opening", "conventional opening or greeting such as hello or hi"),
    "qrr": ("Or-Clause", "or-clause offering alternatives"),
    "arp_nd": ("Dispreferred Answers", "dispreferred or evasive answer"),
    "t3": ("3rd-party-talk", "talk about a third party"),
    "oo_co_cc": ("Offers, Options, Commits", "offer, presenting options, or committing to something"),
    "aap_am": ("Maybe/Accept-part", "partial acceptance or maybe"),
    "t1": ("Downplayer", "downplaying or minimizing the importance of something"),
    "bd": ("Self-talk", "self-directed talk or aside"),
    "^g": ("Tag-Question", "tag question such as right?, isn't it?, or you know?"),
    "qw^d": ("Declarative Wh-Question", "declarative form that functions as a wh-question"),
    "fa": ("Apology", "apology"),
    "ft": ("Thanking", "thanking")
}

# Build candidate descriptions and reverse map
CANDIDATES: List[str] = [f"{human}: {desc}" for _, (human, desc) in DA_DEF.items()]
CODE_FROM_DESC: Dict[str, str] = {f"{human}: {desc}": code for code, (human, desc) in DA_DEF.items()}
NAME_FROM_CODE: Dict[str, str] = {code: human for code, (human, _) in DA_DEF.items()}

len(CANDIDATES), CANDIDATES[:3]


(41,
 ['Statement-non-opinion: factual statement providing information',
  'Acknowledge (Backchannel): short acknowledgement such as okay, uh-huh, or I see',
  'Statement-opinion: opinion or subjective statement'])

In [ ]:

print("Loading dataset:", DATASET_NAME, SPLIT)
ds = load_dataset(DATASET_NAME, split=SPLIT, token=HF_TOKEN)
print(ds)

if "prompt" not in ds.column_names:
    raise ValueError(f"`prompt` column not found. Available columns: {ds.column_names}")

# Optional subsample
if MAX_SAMPLES:
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

texts: List[str] = [str(x) if x is not None else "" for x in ds["prompt"]]
print("Examples to classify:", len(texts))


Loading dataset: wmcnicho/StudyChat train
Dataset({
    features: ['prompt', 'response', 'topic', 'messages', 'timestamp', 'chatId', 'userId', 'interactionCount', 'chatTitle', 'chatStartTime', 'chatTotalInteractionCount', 'llm_label', 'semester'],
    num_rows: 16851
})
Examples to classify: 16851


: 

In [ ]:

clf = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE
)

def chunked(it, n):
    for i in range(0, len(it), n):
        yield it[i:i+n]

pred_codes: List[str] = []
pred_names: List[str] = []
pred_scores: List[float] = []

for batch in tqdm(list(chunked(texts, BATCH_SIZE)), total=(len(texts)+BATCH_SIZE-1)//BATCH_SIZE, desc="Classifying prompts"):
    out = clf(
        batch,
        candidate_labels=CANDIDATES,
        multi_label=False,
        hypothesis_template="This is a {}."
    )
    if isinstance(out, dict):
        out = [out]
    for res in out:
        best_desc = res["labels"][0]  # the chosen description
        score = float(res["scores"][0])
        code = CODE_FROM_DESC[best_desc]
        name = NAME_FROM_CODE[code]
        pred_codes.append(code)
        pred_names.append(name)
        pred_scores.append(score)

assert len(pred_codes) == len(texts)
print("Sample predictions:", list(zip(pred_codes[:5], pred_names[:5], pred_scores[:5])))


Device set to use cpu
Classifying prompts:   0%|          | 0/1054 [00:00<?, ?it/s]

In [ ]:

# Full objects with appended predictions
records: List[Dict[str, Any]] = ds.to_list()
for rec, code, name, score in zip(records, pred_codes, pred_names, pred_scores):
    rec["da_label"] = code
    rec["da_name"] = name
    rec["da_score"] = score

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"Saved JSON: {OUT_JSON} (items: {len(records):,})")

# Flat CSV with selected fields
csv_fields = ["chatId","userId","topic","semester","timestamp","chatTitle","chatStartTime","chatTotalInteractionCount","interactionCount","prompt","da_label","da_name","da_score"]
df = pd.DataFrame([{k: rec.get(k, None) for k in csv_fields} for rec in records])
df.to_csv(OUT_CSV, index=False)
print(f"Saved CSV:  {OUT_CSV} (rows: {len(df):,})")

display(df.head(5))
